# GaussCapture · Colab trainer

Trains a 3D Gaussian splat from a `dataset.zip` produced by
`gausscapture colab <project>`.

**Runtime → Change runtime type → GPU.** A T4 is enough.

---

### Why gsplat and not the reference implementation

The original 3D Gaussian Splatting code from Inria is licensed for
**non-commercial research only**. Anything trained with it inherits that
restriction, which would make GaussCapture's MIT licence a promise it cannot
keep. [gsplat](https://github.com/nerfstudio-project/gsplat) is Apache-2.0,
actively maintained, and produces equivalent quality, so it is what this
notebook uses. See `docs/DEPENDENCIES.md` in the repository.


In [ ]:
# Confirm a GPU is attached. Without one the rest of this notebook will not run.
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU.'
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {torch.cuda.get_device_name(0)}')


## 1 · Install gsplat

A prebuilt wheel matching this runtime's torch and CUDA is tried first; it takes
seconds. Building from source is the fallback and takes several minutes.


In [ ]:
import re, subprocess, sys, torch

torch_tag = 'pt' + ''.join(torch.__version__.split('+')[0].split('.')[:2])
cuda_tag = 'cu' + (torch.version.cuda or '').replace('.', '')
index = f'https://docs.gsplat.studio/whl/{torch_tag}{cuda_tag}'
print('trying', index)

wheel = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'gsplat', '--index-url', index],
    capture_output=True, text=True)
if wheel.returncode != 0:
    print('No matching wheel; building from source (this takes a few minutes).')
    !pip install -q ninja
    !pip install -q git+https://github.com/nerfstudio-project/gsplat.git

# simple_trainer.py needs a few extras beyond the library itself.
!pip install -q tyro tqdm imageio[ffmpeg] torchmetrics 'pycolmap>=3.10' scikit-learn opencv-python-headless plyfile

import gsplat
print('gsplat', gsplat.__version__)


## 2 · Upload the dataset

Upload the `dataset.zip` that `gausscapture colab` produced. It already contains
`images/` and `sparse/0/` in the layout every trainer expects, so nothing here
needs converting.


In [ ]:
import pathlib, shutil, zipfile
from google.colab import files

uploaded = files.upload()
archive = next(iter(uploaded))

data = pathlib.Path('/content/dataset')
if data.exists():
    shutil.rmtree(data)
data.mkdir(parents=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(data)

images = sorted((data / 'images').glob('*.jpg'))
sparse = data / 'sparse' / '0'
assert images, 'No images/ in the archive.'
assert sparse.exists(), 'No sparse/0/ in the archive.'
print(f'{len(images)} images, model files: {sorted(p.name for p in sparse.iterdir())}')


## 3 · Train

`mcmc` is gsplat's densification strategy and is the better default for phone
captures, which have uneven coverage. `--use-bilateral-grid` compensates for
residual exposure drift between frames; harmless when exposure was locked, and
worth several dB when it was not.

30,000 steps is the standard budget and takes roughly 20-40 minutes on a T4. Drop
to 7,000 for a quick look — it reaches most of the quality.


In [ ]:
STEPS = 30_000          # 7_000 for a fast preview
DOWNSCALE = 1           # raise to 2 if the GPU runs out of memory

import pathlib
trainer = pathlib.Path('/content/gsplat_examples/simple_trainer.py')
if not trainer.exists():
    !git clone -q --depth 1 https://github.com/nerfstudio-project/gsplat.git /content/gsplat_src
    !cp -r /content/gsplat_src/examples /content/gsplat_examples

result = pathlib.Path('/content/result')
!python {trainer} mcmc \
    --data_dir /content/dataset \
    --data_factor {DOWNSCALE} \
    --result_dir {result} \
    --max_steps {STEPS} \
    --save_ply \
    --disable_viewer \
    --use_bilateral_grid


## 4 · Collect the splat

The `.ply` is the trained Gaussian splat. Drop it into
[SuperSplat](https://superspl.at/editor) or [Spark](https://sparkjs.dev) to view it,
or import it back into GaussCapture with `gausscapture` to preview and export.


In [ ]:
import pathlib, shutil

plys = sorted(pathlib.Path('/content/result').rglob('*.ply'), key=lambda p: p.stat().st_size)
assert plys, 'Training produced no .ply — check the log above.'
splat = plys[-1]                      # the largest is the final model
print(f'{splat}  ({splat.stat().st_size/1e6:.1f} MB)')

bundle = shutil.make_archive('/content/gausscapture_splat', 'zip', splat.parent)
from google.colab import files
files.download(bundle)
